# 3.3 Linear Regression

**Class 4:** which factors are associated with **systolic blood pressure** (a continuous outcome)?

1. Look at the outcome's distribution
2. A **simple** model: age only
3. **Multivariable** models, compared by how well they fit

> ⚠️ **Keep All of Us data in the Workbench.** Don't download, screenshot, or share participant-level data or counts of 1 to 20. Clear outputs before you share or commit this notebook.

In [ ]:
source("aou_setup.R")  # settings, packages, and helper functions

In [ ]:
library(broom)

fa <- prepare_analysis_data()

## Look at the outcome

A histogram shows skew, outliers, or impossible values before we model.

> ⚠️ Plots of participant data are fine **inside** the Workbench. Don't screenshot or share them if they could reveal small groups.

In [ ]:
ggplot(fa, aes(Systolic_BP)) +
  geom_histogram(bins = 30) +
  labs(x = "Systolic blood pressure (mm Hg)", y = "Participants")

## Simple model: age

`lm(Systolic_BP ~ age)` fits a straight line. The **age coefficient** is the average change in systolic BP (mm Hg) per 1-year increase in age.

In [ ]:
m_age <- lm(Systolic_BP ~ age, data = fa)
tidy(m_age, conf.int = TRUE)

In [ ]:
ggplot(fa, aes(age, Systolic_BP)) +
  geom_point(alpha = 0.1) +
  geom_smooth(method = "lm", se = TRUE) +
  labs(x = "Age (years)", y = "Systolic BP (mm Hg)")

**Interpret:** *"Each additional year of age is associated with an average change of β mm Hg in systolic BP."* Over 10 years, that's 10 × β. Is that clinically meaningful?

## Multivariable models

| Model | Predictors |
|---|---|
| Demog | age, gender, race, ethnicity |
| +BMI+LDL | adds BMI and LDL cholesterol |
| +Diabetes | adds diabetes status |

In [ ]:
m0 <- lm(Systolic_BP ~ age + gender + race + ethnicity, data = fa)
m1 <- update(m0, . ~ . + BMI + LDL_Cholesterol)
m2 <- update(m1, . ~ . + diabetes_binary)

bind_rows(Demog = glance(m0), `+BMI+LDL` = glance(m1), `+Diabetes` = glance(m2), .id = "model") %>%
  select(model, r.squared, adj.r.squared, AIC, BIC)

## How to read model fit

- **R²:** the share of the variation in systolic BP the model explains
- **Adjusted R²:** R², penalised for extra predictors. Use it to compare models of different sizes.
- **AIC / BIC:** lower is better. A drop of more than about **2** is evidence the added variables help; BIC penalises extra variables more.

## Discuss

1. How much does adding **BMI and LDL** improve the fit?
2. Does adding **diabetes** improve it further, or not? Should it stay in the model?
3. Look at the coefficients of your chosen model with `tidy(m1, conf.int = TRUE)`, and check its assumptions with `plot(m1)` (linearity, equal variance, influential points).